In [ ]:
import pandas as pd
import numpy as np 
import re, string

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

from gensim.models import Word2Vec

from nltk.corpus import stopwords

from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, GlobalAveragePooling1D, Dense, Dropout

In [40]:
df = pd.read_csv(r"../data/raw/IMDB Dataset.csv")

In [41]:
df = df.drop_duplicates()

In [ ]:
#lowering the text
df["cleaned_review"] = df["review"].str.lower()

In [ ]:
#removing html tags
df["cleaned_review"] = df["cleaned_review"].apply(
    lambda x: re.sub(r"<.*?>", "", x)
)

In [ ]:
#removing puncutation and white space
df["cleaned_review"] = df["cleaned_review"].apply(
    lambda x: x.translate(str.maketrans("", "", string.punctuation))
)
df["cleaned_review"] = df["cleaned_review"].str.replace(r"\s+", " ", regex=True).str.strip()

In [ ]:
#making stop words and removing not...
stop_words = set(stopwords.words("english"))
stop_words.difference_update(["not", "no", "nor"])

df["cleaned_review"] = df["cleaned_review"].apply(
    lambda x: " ".join(word for word in x.split() if word not in stop_words)
)

In [ ]:
#spliting dataset
x = df["cleaned_review"]
y = df["sentiment"]

x_temp, x_test, y_temp, y_test = train_test_split(
    x, y,
    test_size=0.15,
    random_state=42,
    stratify=y
)

x_train, x_val, y_train, y_val = train_test_split(
    x_temp, y_temp,
    test_size=0.1765,
    random_state=42,
    stratify=y_temp
)

print(x_train.shape, x_val.shape, x_test.shape)

(34705,) (7439,) (7438,)


In [ ]:
#encoding the target variable
encoder = LabelEncoder()
y_train_en = encoder.fit_transform(y_train)
y_val_en = encoder.transform(y_val)
y_test_en = encoder.transform(y_test)

In [ ]:
#tokenization the dataset
VOCAB_SIZE = 20000
MAX_LEN = 200

tokenizer = Tokenizer(num_words=VOCAB_SIZE, oov_token="<OOV>")
tokenizer.fit_on_texts(x_train)

In [ ]:
#making sequence of text and padding
x_train_seq = tokenizer.texts_to_sequences(x_train)
x_val_seq = tokenizer.texts_to_sequences(x_val)
x_test_seq = tokenizer.texts_to_sequences(x_test)

x_train_pad = pad_sequences(x_train_seq, maxlen=MAX_LEN, padding="post", truncating="post")
x_val_pad = pad_sequences(x_val_seq, maxlen=MAX_LEN, padding="post", truncating="post")
x_test_pad = pad_sequences(x_test_seq, maxlen=MAX_LEN, padding="post", truncating="post")

print(x_train_pad.shape, x_val_pad.shape, x_test_pad.shape)

(34705, 200) (7439, 200) (7438, 200)


In [51]:
tfidf = TfidfVectorizer(max_features=20000, ngram_range=(1, 2))

x_train_tfidf = tfidf.fit_transform(x_train)
x_val_tfidf = tfidf.transform(x_val)
x_test_tfidf = tfidf.transform(x_test)

print(x_train_tfidf.shape, x_val_tfidf.shape, x_test_tfidf.shape)

(34705, 20000) (7439, 20000) (7438, 20000)


In [ ]:
#logistic regression model

In [83]:
lr_model = LogisticRegression(max_iter=1000)

lr_model.fit(x_train_tfidf, y_train_en)

y_pred_val = lr_model.predict(x_val_tfidf)

In [84]:
y_pred_val

array([1, 1, 0, ..., 1, 0, 0], shape=(7439,))

In [86]:
accuracy = accuracy_score(y_val_en, y_pred_val)
precision = precision_score(y_val_en, y_pred_val)
recall = recall_score(y_val_en, y_pred_val)
f1 = f1_score(y_val_en, y_pred_val)

print("Accuracy :", accuracy)
print("Precision:", precision)
print("Recall   :", recall)
print("F1 Score :", f1)

Accuracy : 0.9009275440247345
Precision: 0.8896982310093653
Recall   : 0.9161532279667828
F1 Score : 0.9027319519598785


In [ ]:
#wordes embedding

In [71]:
word2vec_model = Word2Vec(
    sentences=x_train_seq,
    vector_size=100,
    window=5,
    min_count=2,
    workers=4
)

In [72]:
def review_to_vector(review, model):
    vectors = []

    for word in review:
        if word in model.wv:
            vectors.append(model.wv[word])

    if len(vectors) == 0:
        return np.zeros(model.vector_size)

    return np.mean(vectors, axis=0)

In [73]:
X_train_w2v = np.array([review_to_vector(review, word2vec_model) for review in x_train_seq])

X_val_w2v = np.array([review_to_vector(review, word2vec_model) for review in x_val_seq])

X_test_w2v = np.array([review_to_vector(review, word2vec_model) for review in x_test_seq])

In [77]:
X_test_w2v

array([[ 0.07393111,  0.28544876, -0.45830324, ...,  0.12603708,
         0.13374111, -0.7274273 ],
       [ 0.4341664 ,  0.11176249,  0.12018293, ..., -0.03621661,
         0.18038233, -0.7674127 ],
       [-0.08114112,  0.26421952, -0.38064772, ...,  0.08719824,
        -0.05485665, -0.31069776],
       ...,
       [-0.06057193,  0.12542261, -0.11561773, ..., -0.24683745,
         0.16150811, -0.18290012],
       [ 0.09008031,  0.28423098, -0.06065587, ..., -0.08585294,
         0.2853368 , -0.8215471 ],
       [-0.01361614,  0.28534326, -0.18520208, ..., -0.01112874,
         0.1531434 ,  0.01121553]], shape=(7438, 100), dtype=float32)

In [74]:
print(X_train_w2v.shape)
print(X_val_w2v.shape)
print(X_test_w2v.shape)

(34705, 100)
(7439, 100)
(7438, 100)


In [ ]:
#logistic model for word embedding

w2v_model = LogisticRegression(max_iter=1000)

w2v_model.fit(X_train_w2v, y_train_en)

y_pred_w2v = w2v_model.predict(X_val_w2v)

In [87]:
accuracy_w2v = accuracy_score(y_val_en, y_pred_w2v)
precision_w2v = precision_score(y_val_en, y_pred_w2v)
recall_w2v = recall_score(y_val_en, y_pred_w2v)
f1_w2v = f1_score(y_val_en, y_pred_w2v)

print("Accuracy :", accuracy_w2v)
print("Precision:", precision_w2v)
print("Recall   :", recall_w2v)
print("F1 Score :", f1_w2v)

Accuracy : 0.8533405027557467
Precision: 0.8461740041928721
Recall   : 0.8649879453522636
F1 Score : 0.8554775466949265


In [ ]:
#embedding model

In [89]:
VOCAB_SIZE = 20000
MAX_LEN = 200

In [90]:
embedding_model = Sequential([
    Embedding(
        input_dim=VOCAB_SIZE,
        output_dim=100,
        input_length=MAX_LEN
    ),
    GlobalAveragePooling1D(),
    Dense(64, activation="relu"),
    Dropout(0.3),
    Dense(1, activation="sigmoid")
])

c:\Users\rizwa\.virtualenvs\week_10-oxvINbMJ\Lib\site-packages\keras\src\layers\core\embedding.py:123: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


In [91]:
embedding_model.compile(optimizer="adam",loss="binary_crossentropy",metrics=["accuracy"])

In [93]:
history_embedding = embedding_model.fit(x_train_pad,y_train_en,validation_data=(x_val_pad, y_val_en),
    epochs=10,batch_size=32)

Epoch 1/10
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 30s 26ms/step - accuracy: 0.7927 - loss: 0.4327 - val_accuracy: 0.8730 - val_loss: 0.3031
Epoch 2/10
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 29s 27ms/step - accuracy: 0.8998 - loss: 0.2493 - val_accuracy: 0.8696 - val_loss: 0.3076
Epoch 3/10
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 30s 28ms/step - accuracy: 0.9218 - loss: 0.1994 - val_accuracy: 0.8688 - val_loss: 0.3297
Epoch 4/10
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 30s 27ms/step - accuracy: 0.9361 - loss: 0.1711 - val_accuracy: 0.8835 - val_loss: 0.3094
Epoch 5/10
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 30s 27ms/step - accuracy: 0.9482 - loss: 0.1419 - val_accuracy: 0.8907 - val_loss: 0.3240
Epoch 6/10
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 30s 28ms/step - accuracy: 0.9523 - loss: 0.1310 - val_accuracy: 0.8771 - val_loss: 0.3769
Epoch 7/10
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 31s 28ms/step - accuracy: 0.9571 - loss: 0.1159 - val_accuracy: 0.8867 - val_loss: 0.3756
Epoch 8/10
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 30s 27ms/step - accuracy: 0.9583 -

In [95]:
y_pred_em = embedding_model.predict(x_val_pad)

233/233 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step


In [101]:
y_pred_em

array([1, 1, 0, ..., 1, 0, 0], shape=(7439,))

In [99]:
y_pred_em = (y_pred_em >= 0.5).astype(int).ravel()

In [100]:
accuracy_em = accuracy_score(y_val_en, y_pred_em)
precision_em = precision_score(y_val_en, y_pred_em)
recall_em = recall_score(y_val_en, y_pred_em)
f1_em = f1_score(y_val_en, y_pred_em)

print("Accuracy :", accuracy_em)
print("Precision:", precision_em)
print("Recall   :", recall_em)
print("F1 Score :", f1_em)

Accuracy : 0.869471703185912
Precision: 0.9179782082324455
Recall   : 0.8124832574336994
F1 Score : 0.8620150632371749


In [ ]:
#comparaison

In [102]:
results = {
    "Representation": ["TF-IDF","Word2Vec","Trainable Embedding"],
    "Model": ["Logistic Regression","Logistic Regression","Neural Network"],
    "Accuracy": [0.9009,0.8533,0.8695],
    "Precision": [0.8897,0.8462,0.9180],
    "Recall": [0.9162,0.8650,0.8125],
    "F1 Score": [0.9027,0.8555,0.8620]
}

In [103]:
df_performance = pd.DataFrame(results)

In [104]:
df_performance

,Representation,Model,Accuracy,Precision,Recall,F1 Score
0,TF-IDF,Logistic Regression,0.9009,0.8897,0.9162,0.9027
1,Word2Vec,Logistic Regression,0.8533,0.8462,0.8650,0.8555
2,Trainable Embedding,Neural Network,0.8695,0.9180,0.8125,0.8620
